# Metadatos de las reglas de detección de SigmaHQ

Cuaderno de lectura de la medición `sigma/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda un artículo en preparación. Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/sigma/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
r = leer("reglas.csv")
print(len(r), "reglas ·", r.fp_declarado.sum(), "declaran falsos positivos ·", r.fp_sustantivo.sum(), "con un falso positivo sustantivo")
r.groupby("coleccion")[["fp_declarado", "fp_sustantivo"]].mean().round(2)

## Una figura

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
orden = ["critical", "high", "medium", "low", "informational"]
a.bar(orden, [int((r.nivel == n).sum()) for n in orden]); a.set_title("reglas por nivel")
g = r.groupby("coleccion")[["fp_declarado", "fp_sustantivo"]].mean()
g.plot.bar(ax=b, rot=30); b.set_title("proporción con falsos positivos declarados / sustantivos"); b.set_ylim(0, 1)
plt.tight_layout()